In [ ]:
# Tåg och trafikanalys Stockholm

Importering av bibliotek

Här importerar jag de bibliotek som jag kommer ha användning för detta projekt vilket inkluderar os, csv, requests, matplotlib och datetime.

In [ ]:
# ==============================================================================================================
# Importering av bibliotek
# ==============================================================================================================

import os
import csv
import requests
import matplotlib.pyplot as plt

from datetime import datetime   # Hämtar dagens datum
from datetime import timedelta  # Hämtar morgondagens datum
from getpass import getpass # Hämtar API-nyckeln från användaren om den inte finns i miljövariablerna


print("Alla bibliotek har importerats!")

API-Nyckeln

In [ ]:
# ==============================================================================================================
# API-nyckel
# ==============================================================================================================

API_KEY = "ad1dd6ea9368474cb498c74084e8573e"

print("API-nyckeln har lästs in!")


Importering av API

In [ ]:
def get_station_locations(station_name):
    """
    Söker efter stationer med ResRobot API.
    """

    url = "https://api.resrobot.se/v2.1/location.name"

    params = {
        "input": station_name,
        "maxNo": 10,
        "lang": "sv",
        "format": "json",
        "accessId": API_KEY
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        return data.get("StopLocation", [])

    except requests.exceptions.RequestException as e:

        print(f"Ett API-fel uppstod: {e}")

        return []


def get_departures(stop_id, date_str):
    """
    Hämtar avgångar från vald station.
    """

    url = "https://api.resrobot.se/v2.1/departureBoard"

    # För dagens datum börjar vi ungefär från aktuell tid.
    # För framtida datum börjar vi vid midnatt.

    today = datetime.now().strftime("%Y-%m-%d")

    if date_str == today:
        time_str = datetime.now().strftime("%H:%M")
    else:
        time_str = "00:00"

    params = {
        "id": stop_id,
        "format": "json",
        "accessId": API_KEY,
        "date": date_str,
        "time": time_str,
        "duration": 120,
        "maxJourneys": 40,
        "lang": "sv"
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        return response.json()

    except requests.exceptions.RequestException as e:

        print(f"Ett API-fel uppstod: {e}")

        return {}


print("Importeringen av API:n är klar!")


Föräldrar klass

In [ ]:
# ==============================================================================================================
# Föräldrarklass
# ==============================================================================================================

class Departure:
    """
    Föräldraklass som representerar en avgång.
    """

    def __init__(self, station, destination, date, scheduled_time, realtime_time, realtime_date, line, transport_category, operator):
        
        self.station = station
        self.destination = destination
        self.date = date
        self.scheduled_time = scheduled_time
        self.realtime_time = realtime_time
        self.realtime_date = realtime_date
        self.line = line
        self.transport_category = transport_category
        self.operator = operator


    def get_delay(self):
        """
        Räknar ut försening i minuter.
        """

        # Om realtidsdata saknas
        if not self.realtime_time:
            return None

        # Om datum saknas
        if not self.date:
            return None

        # Om realtidsdatum saknas används planerat datum
        realtime_date = self.realtime_date

        if not realtime_date:
            realtime_date = self.date

        try:

            scheduled_datetime = datetime.strptime(
                f"{self.date} {self.scheduled_time}",
                "%Y-%m-%d %H:%M:%S"
            )

            realtime_datetime = datetime.strptime(
                f"{realtime_date} {self.realtime_time}",
                "%Y-%m-%d %H:%M:%S"
            )

            difference = (
                realtime_datetime - scheduled_datetime
            ).total_seconds() / 60

            # Negativ försening räknas som 0
            return max(0, int(round(difference)))

        except (ValueError, TypeError):

            return None


    def is_delayed(self):
        """
        Kontrollerar om tåget är försenat.
        """

        delay = self.get_delay()

        if delay is None:
            return False

        return delay > 0


# =================================================================================================================

class TrafficAnalyzer:
    """
    Föräldraklass som analyserar avgångarna.
    """

    def __init__(self, departure_list):

        self.departure_list = departure_list


    def get_realtime_departures(self):
        """
        Returnerar bara avgångar som har realtidsinformation.
        """

        return [
            departure
            for departure in self.departure_list
            if departure.get_delay() is not None
        ]


    def find_most_delayed(self):
        """
        Hittar det mest försenade tåget.
        """

        departures = self.get_realtime_departures()

        if not departures:
            return None

        return max(
            departures,
            key=lambda x: x.get_delay()
        )


    def find_least_delayed(self):
        """
        Hittar det tåg som är minst försenat.
        """

        departures = self.get_realtime_departures()

        if not departures:
            return None

        return min(
            departures,
            key=lambda x: x.get_delay()
        )


    def average_delay(self):
        """
        Räknar ut genomsnittlig försening.
        """

        departures = self.get_realtime_departures()

        if not departures:
            return 0

        total = sum(
            x.get_delay()
            for x in departures
        )

        return total / len(departures)


    def calculate_difference(self):
        """
        Räknar ut skillnaden mellan
        största och minsta försening.
        """

        most_delayed = self.find_most_delayed()
        least_delayed = self.find_least_delayed()

        if not most_delayed or not least_delayed:
            return 0

        return (
            most_delayed.get_delay()
            - least_delayed.get_delay()
        )


    def count_delayed(self):
        """
        Räknar antalet försenade avgångar.
        """

        departures = self.get_realtime_departures()

        return sum(
            1
            for departure in departures
            if departure.get_delay() > 0
        )


    def count_on_time(self):
        """
        Räknar antalet avgångar som är i tid.
        """

        departures = self.get_realtime_departures()

        return sum(
            1
            for departure in departures
            if departure.get_delay() == 0
        )




Barnklass

In [ ]:
class TrainDeparture(Departure):
    """
    Barnklass som ärver från Departure.
    """

    def get_status_text(self):
        """
        Returnerar status beroende på försening.
        """

        delay = self.get_delay()

        if delay is None:
            return "Ingen realtidsinformation"

        elif delay == 0:
            return "I tid"

        elif delay <= 5:
            return "Mindre försening"

        else:
            return "Försenad"


Funktioner

In [ ]:
# ==============================================================================================================
# Funktioner
# ==============================================================================================================

#  Ber användaren skriva en station och väljer en station från API-resultatet.
def validate_station():

    while True:

        station_name = input("\nAnge station, exempelvis Stockholm: ")
        station_name = station_name.strip()

        if not station_name:
            print("Du måste ange en station!")
            continue

        results = get_station_locations(station_name)

        if not results:
            print("Kunde inte hitta stationen.")
            print("Försök igen!")
            continue

        print("\nHittade stationer:")
        print("-" * 50)

        for index, station in enumerate(results, start=1):
            print(f"{index}. " f"{station.get('name')} " f"(ID: {station.get('extId')})")


        while True:
            choice = input(f"\nVälj station mellan 1-{len(results)}: ")

            if (choice.isdigit() and 1 <= int(choice) <= len(results)):
                return results[int(choice) - 1]

            else:
                print("Ogiltigt val, försök igen!")

Menyn

In [ ]:
# ==============================================================================================================
# Menyval
# ==============================================================================================================

while True:

    print("\n" + "-" * 35)
    print("Välkommen till Tåg- och trafikanalysen!")
    print("-" * 35)

    print("1. Analysera dagens tågavgångar")
    print("2. Analysera morgondagens tågavgångar")
    print("3. Avsluta programmet")

    val = input("Välj ett alternativ mellan 1-3: ")


    if val == "1":
        # Hämtar dagens datum
        date = datetime.now().strftime("%Y-%m-%d") 

    elif val == "2":
        # Hämta morgondagens datum
        date_tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d") 
        
    elif val == "3":

        print("\nProgrammet avslutas...")

        break

    else:

        print("\nOgiltigt val, vänligen försök igen!\n")


-----------------------------------
Välkommen till Tåg- och trafikanalysen!
-----------------------------------
1. Analysera dagens tågavgångar
2. Analysera morgondagens tågavgångar
3. Avsluta programmet

-----------------------------------
Välkommen till Tåg- och trafikanalysen!
-----------------------------------
1. Analysera dagens tågavgångar
2. Analysera morgondagens tågavgångar
3. Avsluta programmet

Programmet avslutas. Hej då!
